# 33. 绘图结构（Figure / Axes）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 2 / 12 步：掌握 Figure / Axes 绘图骨架**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** Matplotlib 模块入门  →  **本章任务：** 绘图结构（Figure / Axes）  →  **下一步：** 折线图（plot）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

表格里的数字密密麻麻，趋势、差异和异常往往要靠人眼去"找"，效率很低；图表能把它们变成一眼就懂的视觉关系。



## 本章目标

学完本章，你将能够：

- **理解**：理解「绘图结构（Figure / Axes）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「绘图结构（Figure / Axes）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「绘图结构（Figure / Axes）」并读出其中的结论。


## 33.1 适用场景

**背景引入**：表格里的数字密密麻麻，趋势、差异和异常往往要靠人眼去"找"，效率很低；图表能把它们变成一眼就懂的视觉关系。Matplotlib 的 Figure 与 Axes 结构就是画图的根基——它决定了我们能不能精确控制画布、坐标轴、标题与注释，从而画出一张张让读者脱离代码也能读懂结论的正式图表。几乎所有数据工作台项目都从基础图表起步，掌握它，后面的进阶图、报告与汇报才有清晰的地基。 打个比方：`plt.subplots()` 就像先铺一张画布（Figure），再在画布上划出一格坐标系（Axes），折线、散点、柱状都画在这格坐标系里——你调的是画布的大小，修饰的是坐标系的标题与刻度，画的才是数据本身。

所有Matplotlib章节的基础；适合需要精确控制布局、注释和静态导出的场景。


## 33.2 数据结构

任意可转换为一维或二维数值序列的数据。先明确画布、绘图区和数据元素的职责。


## 33.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 figsize 从 (9, 4.2) 改为 (12, 5)，观察画布尺寸变化
2. 将 layout="constrained" 改为 layout="tight"，对比布局效果
3. 修改 width_ratios 为 [3, 1]，说明主副图宽度比例的变化


## 33.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.plot()`、`ax.set()`、`ax.legend()` | 所有Matplotlib章节的基础；适合需要精确控制布局、注释和静态导出的场景。 | 混用多个隐式plt状态导致图画到错误Axes |
| 进阶变体 | `plt.figure()`、`fig.add_gridspec()`、`fig.add_subplot()`、`ax_main.plot()` | 在基础图表上增加分组、注释、布局或交互 | 创建Figure后未保存引用 |
| 关键参数 | `figsize` | 画布尺寸 | 混用多个隐式plt状态导致图画到错误Axes |
| 关键参数 | `dpi` | 显示或导出分辨率 | 创建Figure后未保存引用 |
| 关键参数 | `layout` | 自动布局 | 没有关闭不再使用的图 |
| 关键参数 | `Axes.set` | 集中设置标题和坐标轴 | 混用多个隐式plt状态导致图画到错误Axes |


## 33.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = completed["InvoiceDate"].dt.to_period("M").astype("string")

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(rows["Country"], rows["flow"], values=rows["amount"].abs(), aggfunc="sum")
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 33.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
line = ax.plot(months, sales, marker="o", color="#1a73e8", label="销售额")[0]
ax.set(title="Figure与Axes示例", xlabel="月份", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Axes数量:", len(fig.axes))
print("数据点数量:", len(line.get_xdata()))


### 练一练：把基础折线图改一个数据字段

上面这张折线图画的是**销售额** sales。请你动手改一版：把数据字段从销售额 sales 换成订单量 orders，并把点的样式 marker 改成 "*"。运行后对比两张图，说说订单量的折线走势和销售额有什么不同。

> 提示：前置代码已经算好了 months、sales、orders 等变量，直接复用即可。


In [ ]:
# 请在下方填写代码
change_note = "数据字段 sales->orders，marker->'*'"  # 我改了什么
expected_change = "待填写"  # 我预期会发生什么
observed_change = "运行后填写"  # 我实际看到什么
# 把下面的 ORDERS 换成真正的数据变量（前置单元格里的 orders），并把 marker 改为 "*"


In [ ]:
import matplotlib.pyplot as plt

# 完整答案：数据字段从 sales 改为 orders，并把 marker 改为 "*"
fig2, ax2 = plt.subplots(figsize=(8, 4.2))
line2 = ax2.plot(months, orders, marker="*", color="#e8710a", label="订单量")[0]
ax2.set(title="订单量折线", xlabel="月份", ylabel="订单量（笔）")
ax2.legend(frameon=False)
fig2.tight_layout()
plt.show()


## 33.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(9, 4.2), layout="constrained")
grid = fig.add_gridspec(1, 2, width_ratios=[2, 1])
ax_main = fig.add_subplot(grid[0, 0])
ax_side = fig.add_subplot(grid[0, 1])
ax_main.plot(months, sales, marker="o", color="#1a73e8")
ax_main.set(title="月度趋势", ylabel="万元")
ax_side.barh(regions, online + offline, color="#188038")
ax_side.set(title="区域合计")
plt.show()


## 33.8 参数说明

- figsize：画布尺寸
- dpi：显示或导出分辨率
- layout：自动布局
- Axes.set：集中设置标题和坐标轴


### 参数示例：dpi 与 Axes.set

下面给两个独立、可直接运行的示例：一个用 `dpi` 控制显示/导出分辨率，一个用 `Axes.set` 集中设置标题与坐标轴标签。


In [ ]:
# 中文字体支持：避免图表中文显示为方框
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 示例 1：用 dpi 控制分辨率（更高 dpi = 导出的图更清晰）
fig, ax = plt.subplots(figsize=(6, 3), dpi=120)
ax.plot([1, 2, 3], [4, 5, 6])
ax.set(xlabel="月份", ylabel="金额")
ax.set_title("dpi=120 的示例图")
print("figsize=(6,3), dpi=120 -> 导出更清晰")
plt.close(fig)

# 示例 2：用 Axes.set 集中设置标题与坐标轴标签
fig2, ax2 = plt.subplots()
ax2.set(title="用 set 集中设置", xlabel="X", ylabel="Y")
print("Axes.set 可集中设置 title/xlabel/ylabel")
plt.close(fig2)


## 33.9 结果解读

检查画布中Axes数量、每个Axes的数据对象数量，以及标题、单位是否完整。


## 33.10 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月", "4月"]
sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 33.10.1 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 33.10.2 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 33.11 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

months = ["1月", "2月", "3月"]
sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(months, sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 33.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 33.12 易错点提醒

- 混用多个隐式plt状态导致图画到错误Axes
- 创建Figure后未保存引用
- 没有关闭不再使用的图


## 33.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 33.14 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把「销售额」折线换成「利润」字段，回答新问题
# 【目标】练习迁移能力：同一张图，换一个数据字段，问题也随之改变。
#   之前画的是「销售额走势」，现在回答「利润走势」——观察两者是否同步波动。
import matplotlib.pyplot as plt

# 起点示例（已可运行）：只把 y 从 sales 换成 profit，其余保持不变。
#   - marker="s" 用方块标记，与折线的圆点区分，便于识别；
#   - 颜色换绿色系，避免和上一张蓝色销售额混淆。
fig, ax = plt.subplots(figsize=(8, 4.2))
line = ax.plot(months, profit, marker="s", color="#188038", label="利润")[0]
ax.set(title="上半年利润走势", xlabel="月份", ylabel="利润（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# ---- 反思记录：改一个字段后，用三句话把观察写下来 ----
# ① 我改了什么？（换成了哪个字段/编码）
change_note = "待填写"
# ② 我预期图会怎么变？（走势更陡/更平/方向相反）
expected_change = "待填写"
# ③ 运行后实际观察到什么？和预期一致吗？
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].plot(months, profit, marker="o", color="#188038")
axes[0].set(title="月度利润", ylabel="万元")
axes[1].bar(regions, online, color="#1a73e8")
axes[1].set(title="区域线上销售", ylabel="万元")
fig.tight_layout()
plt.show()


## 33.15 小结

理解Figure、Axes和Artist层级，建立可维护的Matplotlib绘图流程。


### 33.15.1 你已经掌握

- 判断绘图结构（Figure / Axes）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 33.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `figsize` | 画布尺寸 |
| `dpi` | 显示或导出分辨率 |
| `layout` | 自动布局 |
| `Axes.set` | 集中设置标题和坐标轴 |


### 33.15.3 需要注意

- 混用多个隐式plt状态导致图画到错误Axes
- 创建Figure后未保存引用
- 没有关闭不再使用的图


### 33.15.4 完成检查

- [ ] 能判断什么问题适合使用绘图结构（Figure / Axes）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 33.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
